# Step 24C — 288px cache, Axial

**Settings: Accelerator = `None` (CPU), Internet = `Off`.**
**Inputs:** competition data, `rsna-knee-12-laterality`

**Writes `cache288_c.npy`, 11.7 GB.** ~20 min, zero GPU cost.

## Why three notebooks

At 288px with 32 slices, one plane is 11.7 GB and all three would be 35.1 GB — past Kaggle's 20 GB per-notebook output limit. So each plane gets its own notebook, and training attaches all three and concatenates along the series axis.

## Why 288px

Resolution has been the most reliable lever in this competition: 160 -> 224 gave **+0.020** on the leaderboard, larger than anything else tried.

And your own numbers say where the remaining headroom is. Comparing each label against the quality of its own training labels:

| | extraction | model | gap |
|---|---|---|---|
| Lateral Meniscus | 0.761 | 0.652 | **-0.109** |
| Effusion | 0.793 | 0.746 | -0.047 |

Lateral Meniscus sits far below its own label quality — the information is in the labels and the model can't see it in the pixels. A lateral meniscus is a few millimetres; at 224px it spans perhaps 4-5 pixels, at 288px around 6-7. Effusion, by contrast, is near its label ceiling, so resolution won't help there.

Expect less than the 160->224 jump — diminishing returns as pixel size approaches the anatomy — but it's the lever with the track record.

## Unchanged from the 224px cache

Same series selection, same geometric slice ordering, same per-series percentile normalisation, same laterality resolution from `laterality.csv`. Only `PX` changes.

### 1 · Setup

In [ ]:
import os, glob, time, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, cv2, pydicom
from multiprocessing import Pool

ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
OUT  = "/kaggle/working"

PX, NSL = 288, 32
PLANE   = "Axial"
LAT_THR, LAT_SIGN = -37.5, 1

train = pd.read_csv(f"{ROOT}/train.csv")
ser   = pd.read_csv(f"{ROOT}/train_series.csv")
studies = train["StudyInstanceUID"].tolist()

lat_df = pd.read_csv(glob.glob("/kaggle/input/notebooks/*/*/laterality.csv")[0]).set_index("StudyInstanceUID")
LAT = lat_df["lat_final"].to_dict()
print("plane:", PLANE)
print(lat_df.lat_src.value_counts().to_string())
print(f"\ncache ({len(studies)}, 1, {NSL}, {PX}, {PX}) = "
      f"{len(studies)*NSL*PX*PX/1e9:.1f} GB   (limit 20)")

### 2 · Series selection

In [ ]:
def pick(g):
    sub = g[g["Anatomical_Plane"] == PLANE]
    if len(sub) == 0:
        return None
    sub = sub.sort_values(["Fluid_Sensitive", "Fat_Suppression"], ascending=False)
    return sub.iloc[0]["SeriesInstanceUID"]

chosen = {suid: pick(g) for suid, g in ser.groupby("StudyInstanceUID")}
have = np.array([chosen.get(s) is not None for s in studies])
print(f"{PLANE} present in {have.mean():.1%} of studies")

### 3 · Decoder — identical to the 224px cache apart from PX

In [ ]:
def resolve_lat(suid, metas):
    l = str(LAT.get(suid, "") or "").strip().upper()
    if l in ("L", "R"):
        return l
    tag = next((m[4] for m in metas if m[4]), "")
    if tag:
        return tag.strip().upper()[0]
    xs = [float(m[2][0]) for m in metas if m[2] is not None]
    if xs:
        return "L" if (np.mean(xs) - LAT_THR) * LAT_SIGN > 0 else "R"
    return "R"


def load_series(dirpath, suid):
    files = os.listdir(dirpath)
    if not files: return None
    metas = []
    for f in files:
        try:
            d = pydicom.dcmread(os.path.join(dirpath, f), stop_before_pixels=True)
            iop = getattr(d, "ImageOrientationPatient", None)
            ipp = getattr(d, "ImagePositionPatient", None)
            metas.append((f, np.array(iop, float) if iop else None,
                             np.array(ipp, float) if ipp else None,
                             getattr(d, "InstanceNumber", 0),
                             str(getattr(d, "Laterality", "") or "")))
        except Exception:
            continue
    if not metas: return None

    iops = [m[1] for m in metas if m[1] is not None]
    if iops:
        iop = iops[0]; r, c = iop[:3], iop[3:]; n = np.cross(r, c)
        pos = [(float(np.dot(m[2], n)) if m[2] is not None else float(m[3])) for m in metas]
    else:
        r, c, n = np.array([1.,0,0]), np.array([0,1.,0]), np.array([0,0,1.])
        pos = [float(m[3]) for m in metas]

    order = np.argsort(pos); metas = [metas[i] for i in order]
    take = np.linspace(0, len(metas)-1, NSL).round().astype(int)

    imgs = []
    for i in take:
        try:
            d = pydicom.dcmread(os.path.join(dirpath, metas[i][0]))
            a = d.pixel_array.astype(np.float32)
            if hasattr(d, "RescaleSlope"):
                a = a * float(d.RescaleSlope) + float(getattr(d, "RescaleIntercept", 0))
        except Exception:
            a = np.zeros((PX, PX), np.float32)
        imgs.append(cv2.resize(a, (PX, PX), interpolation=cv2.INTER_AREA))
    vol = np.stack(imgs)

    lo, hi = np.percentile(vol, [1, 99])
    if hi <= lo: hi = lo + 1
    vol = np.clip((vol - lo) / (hi - lo), 0, 1)

    axes = [n, c, r]
    for ax, vec in enumerate(axes):
        dom = int(np.argmax(np.abs(vec)))
        if vec[dom] < 0: vol = np.flip(vol, axis=ax)
        axes[ax] = vec * (1 if vec[dom] >= 0 else -1)

    if resolve_lat(suid, metas) == "L":
        vol = np.flip(vol, axis=int(np.argmax([abs(v[0]) for v in axes])))

    return (vol * 255).astype(np.uint8)

print("decoder ready")

### 4 · Build

Resumable via `done288_c.npy`.

In [ ]:
CACHE = f"{OUT}/cache288_c.npy"
DONE  = f"{OUT}/done288_c.npy"
SHAPE = (len(studies), 1, NSL, PX, PX)

if not os.path.exists(CACHE):
    np.lib.format.open_memmap(CACHE, mode="w+", dtype=np.uint8, shape=SHAPE)
    np.save(DONE, np.zeros(len(studies), np.uint8))
    print("created", CACHE)

done = np.load(DONE)
print(f"already done: {done.sum():,}/{len(studies):,}")

def work(i):
    a = np.lib.format.open_memmap(CACHE, mode="r+")
    suid = studies[i]; got = 0
    cs = chosen.get(suid)
    if cs:
        p = f"{ROOT}/train_series/{suid}/{cs}"
        if os.path.isdir(p):
            try:
                v = load_series(p, suid)
                if v is not None:
                    a[i, 0] = v; got = 1
            except Exception:
                pass
    del a
    return i, got

todo = [i for i in range(len(studies)) if not done[i]]
print(f"to process: {len(todo):,}\n")

t0 = time.time()
with Pool(4) as pool:
    for n, (i, got) in enumerate(pool.imap_unordered(work, todo, chunksize=8), 1):
        done[i] = 1
        if n % 400 == 0 or n == len(todo):
            np.save(DONE, done)
            el = time.time() - t0
            print(f"  {n}/{len(todo)}  {el/n:.2f}s/study  eta {(len(todo)-n)*el/n/60:.0f} min", flush=True)
np.save(DONE, done)
print(f"\ncomplete: {done.sum():,}/{len(studies):,} in {(time.time()-t0)/60:.0f} min")

### 5 · Verify

In [ ]:
import matplotlib.pyplot as plt
arr = np.load(CACHE, mmap_mode="r")
rng = np.random.default_rng(0)
chk = rng.choice(len(studies), 300, replace=False)

nz = np.array([arr[i, 0].max() > 0 for i in chk])
print("shape", arr.shape)
print(f"{PLANE} populated: {nz.mean():.1%}")
print(f"intensity mean {arr[chk[:30]].mean():.1f} std {arr[chk[:30]].std():.1f}")

i = int(np.where(done > 0)[0][0])
fig, ax = plt.subplots(1, 8, figsize=(18, 2.6))
for j in range(8):
    ax[j].imshow(arr[i, 0, j*4], cmap="gray", vmin=0, vmax=255)
    ax[j].axis("off")
plt.suptitle(f"288px {PLANE} - every 4th slice")
plt.tight_layout(); plt.show()